# ПР-12. Валидация и подбор гиперпараметров

Построить честный процесс сравнения: отложить тест один раз, подобрать гиперпараметры только на train и сохранить тест для финальной оценки.

**Данные:** см. README проекта. Выполняйте ячейки сверху вниз; фиксируйте результаты в отчёте.

## 1. Подготовка данных

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
data = pd.read_csv(ROOT / 'data' / 'classification_breast_cancer.csv')
X = data.drop(columns=['target', 'target_name'])
y = data['target'].astype(int)  # 0 = malignant, 1 = benign
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print('train/test:', X_train.shape, X_test.shape)


## 1. Поиск по CV

Сравним Logistic Regression и Random Forest по macro F1. Pipeline не допускает утечку статистик масштабирования между фолдами.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold,GridSearchCV
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
searches={
 'Logistic Regression':GridSearchCV(Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=5000,random_state=42))]),
    {'model__C':[.01,.1,1,10,100]},scoring='f1_macro',cv=cv,n_jobs=-1,return_train_score=True),
 'Random Forest':GridSearchCV(RandomForestClassifier(random_state=42),
    {'n_estimators':[100,300],'max_depth':[None,5,10],'min_samples_leaf':[1,3]},
    scoring='f1_macro',cv=cv,n_jobs=-1,return_train_score=True)}
for name,s in searches.items():
    s.fit(X_train,y_train); print(name,s.best_params_,round(s.best_score_,3))

## 2. Финальная оценка

Только после фиксации выбранных параметров используйте тест. Не меняйте модель после изучения тестового результата.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
rows=[]
for name,s in searches.items():
    pred=s.predict(X_test)
    rows.append({'model':name,'CV F1 macro':s.best_score_,'test accuracy':accuracy_score(y_test,pred),
                 'test F1 macro':f1_score(y_test,pred,average='macro')})
display(pd.DataFrame(rows).set_index('model').round(3))
for name,s in searches.items():
    print('\n',name); print(classification_report(y_test,s.predict(X_test),target_names=['malignant','benign']))
    ConfusionMatrixDisplay.from_predictions(y_test,s.predict(X_test),display_labels=['malignant','benign'],cmap='Blues')
    plt.title(name); plt.show()

## Контрольные вопросы и вывод

1. Почему тест разделяется до подбора?
2. Зачем StandardScaler находится внутри Pipeline?
3. Почему после многократной настройки по тесту он перестаёт быть независимой оценкой?
4. Назовите два возможных объяснения разрыва CV и test.

**В отчёте:** CV и test метрики, лучшие гиперпараметры, матрицы ошибок, объяснение возможного разрыва и ограничения метрики.